# AI-Assisted Diagnosis of Cutaneous Melanoma
## Standard U-Net vs Attention U-Net vs ResUNet → ROI → EfficientNetB0 → Grad-CAM → Gradio

This notebook is a clean final comparison pipeline.

**Core design:** dermoscopic image → preprocessing → compare segmentation architectures → select best segmenter on validation data → predicted lesion mask → lesion ROI → EfficientNetB0 classification → confusion matrix and clinical metrics → Grad-CAM → Gradio app.

### Segmentation models
- Standard U-Net
- Attention U-Net
- ResUNet
- U-Net++ placeholder for Daisy's real team results

### Dataset modes
- `tutor_200`: 200 stratified cases → 100 modelling (80 train + 20 validation) + 100 untouched final test.
- `full_900`: all 900 Task 3B training cases → internal 70/15/15 split.

The final test split is not used for model selection or threshold optimisation.

## 0. How to run this notebook

1. In Google Colab choose **Runtime → Change runtime type → T4 GPU**.
2. Run the notebook from top to bottom.
3. The official ISIC 2016 Task 3B ZIP is downloaded to temporary Colab storage (`/content`) only.
4. Only the selected 200 image/mask pairs are extracted.
5. TensorFlow `tf.data` reads images in batches during training rather than loading the whole dataset into RAM.
6. Google Drive is optional and is used only for small persistent outputs such as models, split CSVs and result tables.

## 1. Environment setup

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("gradio") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "gradio>=4.0"]
    )

print("Environment setup complete.")

## 2. Imports and reproducibility

In [ ]:
import os
import gc
import json
import random
import shutil
import time
import zipfile
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    average_precision_score,
    precision_recall_curve,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 3. Global configuration

In [ ]:
# ==========================================
# GLOBAL CONFIGURATION
# ==========================================
SEED = 42

DATA_MODE = "tutor_200"   # "tutor_200" or "full_900"

TUTOR_TOTAL = 200
TUTOR_MODELLING = 100
TUTOR_TEST = 100
TUTOR_TRAIN = 80
TUTOR_VAL = 20

FULL_TRAIN_FRACTION = 0.70
FULL_VAL_FRACTION = 0.15
FULL_TEST_FRACTION = 0.15

SEG_IMAGE_SIZE = (256, 256)
SEG_BATCH_SIZE = 4
SEG_EPOCHS = 35
SEG_LEARNING_RATE = 1e-3
SEG_THRESHOLD = 0.50

CLS_IMAGE_SIZE = (224, 224)
CLS_BATCH_SIZE = 16
CLS_HEAD_EPOCHS = 12
CLS_FINE_TUNE_EPOCHS = 8
CLS_HEAD_LR = 1e-3
CLS_FINE_TUNE_LR = 1e-5
DEFAULT_CLS_THRESHOLD = 0.50

DO_FINE_TUNING = True
OPTIMISE_THRESHOLD_ON_VALIDATION = True
RUN_CLASSIFIER_BASELINE = False

RUNTIME_ROOT = Path("/content/melanoma_resunet_project")
DOWNLOAD_DIR = RUNTIME_ROOT / "downloads"
DATA_DIR = RUNTIME_ROOT / "selected_data"
ROI_DIR = RUNTIME_ROOT / "roi_images"

SAVE_RESULTS_TO_DRIVE = True

for folder in [RUNTIME_ROOT, DOWNLOAD_DIR, DATA_DIR, ROI_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

assert DATA_MODE in {"tutor_200", "full_900"}

CONFIG = {
    "data_mode": DATA_MODE,
    "seed": SEED,
    "seg_image_size": list(SEG_IMAGE_SIZE),
    "seg_batch_size": SEG_BATCH_SIZE,
    "seg_epochs": SEG_EPOCHS,
    "seg_learning_rate": SEG_LEARNING_RATE,
    "seg_threshold": SEG_THRESHOLD,
    "cls_image_size": list(CLS_IMAGE_SIZE),
    "cls_batch_size": CLS_BATCH_SIZE,
    "cls_head_epochs": CLS_HEAD_EPOCHS,
    "cls_fine_tune_epochs": CLS_FINE_TUNE_EPOCHS,
    "cls_head_lr": CLS_HEAD_LR,
    "cls_fine_tune_lr": CLS_FINE_TUNE_LR,
    "default_cls_threshold": DEFAULT_CLS_THRESHOLD,
    "optimise_threshold_on_validation": OPTIMISE_THRESHOLD_ON_VALIDATION,
    "run_classifier_baseline": RUN_CLASSIFIER_BASELINE,
}

print(json.dumps(CONFIG, indent=2))

## 4. Optional Google Drive output folder

In [ ]:
if SAVE_RESULTS_TO_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        OUTPUT_DIR = Path(
            "/content/drive/MyDrive/Melanoma_AI_200_Case_Results"
        )
    except Exception as exc:
        print("Drive mount unavailable. Falling back to /content.")
        print("Reason:", exc)
        OUTPUT_DIR = RUNTIME_ROOT / "results"
else:
    OUTPUT_DIR = RUNTIME_ROOT / "results"

MODELS_DIR = OUTPUT_DIR / "models"
RESULTS_DIR = OUTPUT_DIR / "results"
SPLITS_DIR = OUTPUT_DIR / "splits"

for folder in [OUTPUT_DIR, MODELS_DIR, RESULTS_DIR, SPLITS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Persistent output directory:", OUTPUT_DIR)

(RESULTS_DIR / "pipeline_config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")


## 5. Official ISIC 2016 Task 3B data

The official **ISIC 2016 Task 3B: Segmented Lesion Classification** training data contains:

- 900 dermoscopic JPEG images,
- 900 matching expert segmentation masks,
- 900 benign/malignant labels.

This notebook downloads the official training ZIP and CSV into temporary Colab storage, selects a stratified 200-case subset, extracts only those selected cases, and deletes the large ZIP afterwards.

In [ ]:
DATA_ZIP_URL = (
    "https://isic-archive.s3.amazonaws.com/challenges/2016/"
    "ISBI2016_ISIC_Part3B_Training_Data.zip"
)

LABEL_CSV_URL = (
    "https://isic-archive.s3.amazonaws.com/challenges/2016/"
    "ISBI2016_ISIC_Part3B_Training_GroundTruth.csv"
)

DATA_ZIP = DOWNLOAD_DIR / "ISBI2016_ISIC_Part3B_Training_Data.zip"
LABEL_CSV = DOWNLOAD_DIR / "ISBI2016_ISIC_Part3B_Training_GroundTruth.csv"


def download_file(url, destination, chunk_mb=8):
    destination = Path(destination)

    if destination.exists() and destination.stat().st_size > 100:
        print("Already downloaded:", destination.name)
        return destination

    tmp = destination.with_suffix(destination.suffix + ".part")
    start = tmp.stat().st_size if tmp.exists() else 0
    headers = {"Range": f"bytes={start}-"} if start else {}

    print("Downloading:", destination.name)

    with requests.get(
        url,
        stream=True,
        headers=headers,
        timeout=(30, 120),
    ) as response:
        response.raise_for_status()

        if start and response.status_code != 206:
            start = 0
            tmp.unlink(missing_ok=True)

        mode = "ab" if start else "wb"

        with open(tmp, mode) as handle:
            downloaded = start
            last_print = time.time()

            for chunk in response.iter_content(
                chunk_size=chunk_mb * 1024 * 1024
            ):
                if not chunk:
                    continue

                handle.write(chunk)
                downloaded += len(chunk)

                if time.time() - last_print > 5:
                    print(
                        f"  downloaded {downloaded / (1024**2):.1f} MB"
                    )
                    last_print = time.time()

    tmp.replace(destination)
    print("Finished:", destination)
    return destination


download_file(LABEL_CSV_URL, LABEL_CSV)
download_file(DATA_ZIP_URL, DATA_ZIP)

assert LABEL_CSV.exists(), "Ground-truth CSV download failed."
assert DATA_ZIP.exists(), "Training ZIP download failed."
assert zipfile.is_zipfile(DATA_ZIP), (
    "Downloaded training file is not a valid ZIP."
)

print("Official ISIC files are ready in temporary Colab storage.")

## 6. Load all labels and select exactly 200 cases

In [ ]:
labels_df = pd.read_csv(
    LABEL_CSV,
    header=None,
    names=["image_id", "diagnosis"],
)

labels_df["image_id"] = labels_df["image_id"].astype(str).str.strip()
labels_df["diagnosis"] = labels_df["diagnosis"].astype(str).str.strip().str.lower()
labels_df["label"] = labels_df["diagnosis"].map({"benign": 0, "malignant": 1})

assert len(labels_df) == 900, f"Expected 900 labels; found {len(labels_df)}."
assert labels_df["image_id"].is_unique, "Duplicate image IDs detected."
assert labels_df["label"].notna().all(), "Unexpected diagnosis values found."

print("Full Task 3B distribution:")
display(labels_df["diagnosis"].value_counts().to_frame("count"))

if DATA_MODE == "tutor_200":
    selected_df, unused_df = train_test_split(
        labels_df,
        train_size=TUTOR_TOTAL,
        stratify=labels_df["label"],
        random_state=SEED,
    )
    selected_df = selected_df.reset_index(drop=True)
else:
    selected_df = labels_df.copy().reset_index(drop=True)

print("Selected mode:", DATA_MODE)
print("Selected cases:", len(selected_df))
display(selected_df["diagnosis"].value_counts().to_frame("count"))

## 7. Create the tutor-directed 100 modelling / 100 testing split

In [ ]:
if DATA_MODE == "tutor_200":
    modelling_df, test_df = train_test_split(
        selected_df,
        train_size=TUTOR_MODELLING,
        test_size=TUTOR_TEST,
        stratify=selected_df["label"],
        random_state=SEED,
    )

    train_df, val_df = train_test_split(
        modelling_df,
        train_size=TUTOR_TRAIN,
        test_size=TUTOR_VAL,
        stratify=modelling_df["label"],
        random_state=SEED,
    )
else:
    train_df, temp_df = train_test_split(
        selected_df,
        train_size=FULL_TRAIN_FRACTION,
        stratify=selected_df["label"],
        random_state=SEED,
    )

    relative_test = FULL_TEST_FRACTION / (FULL_VAL_FRACTION + FULL_TEST_FRACTION)

    val_df, test_df = train_test_split(
        temp_df,
        test_size=relative_test,
        stratify=temp_df["label"],
        random_state=SEED,
    )
    modelling_df = pd.concat([train_df, val_df], ignore_index=True)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
modelling_df = modelling_df.reset_index(drop=True)

assert set(train_df["image_id"]).isdisjoint(set(val_df["image_id"]))
assert set(train_df["image_id"]).isdisjoint(set(test_df["image_id"]))
assert set(val_df["image_id"]).isdisjoint(set(test_df["image_id"]))

def show_split(name, frame):
    print(f"\n{name}: n={len(frame)}")
    counts = frame["diagnosis"].value_counts()
    pct = frame["diagnosis"].value_counts(normalize=True).mul(100).round(1)
    display(pd.DataFrame({"count": counts, "percent": pct}))

show_split("TRAIN", train_df)
show_split("VALIDATION", val_df)
show_split("FINAL TEST", test_df)

train_df[["image_id", "diagnosis", "label"]].to_csv(SPLITS_DIR / "train_split.csv", index=False)
val_df[["image_id", "diagnosis", "label"]].to_csv(SPLITS_DIR / "validation_split.csv", index=False)
test_df[["image_id", "diagnosis", "label"]].to_csv(SPLITS_DIR / "test_split.csv", index=False)

print("Final test data are locked and not used for model selection.")

## 8. Extract only the selected 200 images and masks

The 900-case ZIP is **not** extracted in full. Only 200 selected images and 200 matching masks are extracted.

After successful extraction, the large ZIP is deleted from temporary Colab storage.

In [ ]:
selected_ids = set(selected_df["image_id"])

with zipfile.ZipFile(DATA_ZIP, "r") as archive:
    file_members = [
        member
        for member in archive.namelist()
        if not member.endswith("/")
    ]

    by_basename = {
        Path(member).name: member
        for member in file_members
    }

    missing_images = []
    missing_masks = []
    extracted_records = []

    for image_id in sorted(selected_ids):
        image_base = f"{image_id}.jpg"
        mask_base = f"{image_id}_Segmentation.png"

        image_member = by_basename.get(image_base)
        mask_member = by_basename.get(mask_base)

        if image_member is None:
            missing_images.append(image_id)
            continue

        if mask_member is None:
            missing_masks.append(image_id)
            continue

        image_output = DATA_DIR / image_base
        mask_output = DATA_DIR / mask_base

        if not image_output.exists():
            with archive.open(image_member) as src:
                with open(image_output, "wb") as dst:
                    shutil.copyfileobj(src, dst)

        if not mask_output.exists():
            with archive.open(mask_member) as src:
                with open(mask_output, "wb") as dst:
                    shutil.copyfileobj(src, dst)

        extracted_records.append(
            {
                "image_id": image_id,
                "image_path": str(image_output),
                "mask_path": str(mask_output),
            }
        )

assert not missing_images, (
    f"Missing selected images: {missing_images[:10]}"
)
assert not missing_masks, (
    f"Missing selected masks: {missing_masks[:10]}"
)
assert len(extracted_records) == 200, (
    f"Expected 200 extracted pairs, found "
    f"{len(extracted_records)}."
)

path_df = pd.DataFrame(extracted_records)


def add_paths(frame):
    result = frame.merge(
        path_df,
        on="image_id",
        how="left",
    )
    assert result["image_path"].notna().all()
    assert result["mask_path"].notna().all()
    return result


train_df = add_paths(train_df)
val_df = add_paths(val_df)
model_df = add_paths(model_df)
test_df = add_paths(test_df)
selected_df = add_paths(selected_df)

if DATA_ZIP.exists():
    DATA_ZIP.unlink()
    print(
        "Deleted large temporary ZIP:",
        DATA_ZIP.name,
    )

print(
    "Extracted selected image-mask pairs:",
    len(selected_df),
)
print("Selected data directory:", DATA_DIR)

## 9. Dataset sanity checks and visualisation

In [ ]:
def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def load_mask(path):
    with Image.open(path) as image:
        return image.convert("L").copy()


sample_rows = pd.concat(
    [
        train_df[train_df["label"] == 0].head(2),
        train_df[train_df["label"] == 1].head(2),
    ]
).reset_index(drop=True)

fig, axes = plt.subplots(
    len(sample_rows),
    3,
    figsize=(12, 4 * len(sample_rows)),
)

if len(sample_rows) == 1:
    axes = np.array([axes])

for row_index, row in sample_rows.iterrows():
    image = load_rgb(row["image_path"])
    mask = load_mask(row["mask_path"])

    axes[row_index, 0].imshow(image)
    axes[row_index, 0].set_title(
        f"{row['image_id']} | {row['diagnosis']}"
    )
    axes[row_index, 0].axis("off")

    axes[row_index, 1].imshow(mask, cmap="gray")
    axes[row_index, 1].set_title("Expert mask")
    axes[row_index, 1].axis("off")

    axes[row_index, 2].imshow(image)
    axes[row_index, 2].imshow(
        mask,
        cmap="Reds",
        alpha=0.30,
    )
    axes[row_index, 2].set_title("Mask overlay")
    axes[row_index, 2].axis("off")

plt.tight_layout()
plt.show()

# PART A — LESION SEGMENTATION

We compare:

1. **Standard U-Net**
2. **Attention U-Net**

Both models use the same:

- 80 training cases,
- 20 validation cases,
- image size,
- filter schedule,
- loss,
- optimiser,
- batch size,
- and callbacks.

The main difference is the attention gating of the encoder-to-decoder skip connections.

## 10. `tf.data` segmentation pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_segmentation_pair(
    image_path,
    mask_path,
):
    image_bytes = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(
        image_bytes,
        channels=3,
    )
    image = tf.image.resize(
        image,
        SEG_IMAGE_SIZE,
        method="bilinear",
    )
    image = tf.cast(image, tf.float32) / 255.0

    mask_bytes = tf.io.read_file(mask_path)
    mask = tf.io.decode_png(
        mask_bytes,
        channels=1,
    )
    mask = tf.image.resize(
        mask,
        SEG_IMAGE_SIZE,
        method="nearest",
    )
    mask = tf.cast(mask > 127, tf.float32)

    return image, mask


def paired_segmentation_augmentation(
    image,
    mask,
):
    seed1 = tf.random.uniform(
        [2],
        maxval=2**31 - 1,
        dtype=tf.int32,
    )
    seed2 = tf.random.uniform(
        [2],
        maxval=2**31 - 1,
        dtype=tf.int32,
    )

    image = tf.image.stateless_random_flip_left_right(
        image,
        seed1,
    )
    mask = tf.image.stateless_random_flip_left_right(
        mask,
        seed1,
    )

    image = tf.image.stateless_random_flip_up_down(
        image,
        seed2,
    )
    mask = tf.image.stateless_random_flip_up_down(
        mask,
        seed2,
    )

    image = tf.image.random_brightness(
        image,
        max_delta=0.06,
    )
    image = tf.image.random_contrast(
        image,
        lower=0.90,
        upper=1.10,
    )
    image = tf.clip_by_value(
        image,
        0.0,
        1.0,
    )

    return image, mask


def make_seg_dataset(
    frame,
    training=False,
):
    ds = tf.data.Dataset.from_tensor_slices(
        (
            frame["image_path"].astype(str).values,
            frame["mask_path"].astype(str).values,
        )
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    ds = ds.map(
        decode_segmentation_pair,
        num_parallel_calls=AUTOTUNE,
    )

    if training:
        ds = ds.map(
            paired_segmentation_augmentation,
            num_parallel_calls=AUTOTUNE,
        )

    ds = ds.batch(SEG_BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds


train_seg_ds = make_seg_dataset(
    train_df,
    training=True,
)
val_seg_ds = make_seg_dataset(
    val_df,
    training=False,
)
test_seg_ds = make_seg_dataset(
    test_df,
    training=False,
)

images_batch, masks_batch = next(
    iter(train_seg_ds)
)

print(
    "Image batch shape:",
    images_batch.shape,
)
print(
    "Mask batch shape:",
    masks_batch.shape,
)
print(
    "Image range:",
    float(tf.reduce_min(images_batch)),
    "to",
    float(tf.reduce_max(images_batch)),
)
print(
    "Mask values:",
    np.unique(masks_batch.numpy()),
)

## 11. Dice, IoU/Jaccard and segmentation loss

In [ ]:
EPS = 1e-7


@keras.utils.register_keras_serializable(
    package="MelanomaAI"
)
def dice_coefficient(y_true, y_pred):
    y_true = tf.cast(
        y_true,
        tf.float32,
    )
    y_pred = tf.cast(
        y_pred,
        tf.float32,
    )

    y_true_f = tf.reshape(
        y_true,
        [-1],
    )
    y_pred_f = tf.reshape(
        y_pred,
        [-1],
    )

    intersection = tf.reduce_sum(
        y_true_f * y_pred_f
    )

    return (
        2.0 * intersection + EPS
    ) / (
        tf.reduce_sum(y_true_f)
        + tf.reduce_sum(y_pred_f)
        + EPS
    )


@keras.utils.register_keras_serializable(
    package="MelanomaAI"
)
def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(
        y_true,
        y_pred,
    )


@keras.utils.register_keras_serializable(
    package="MelanomaAI"
)
def iou_coefficient(y_true, y_pred):
    y_true = tf.cast(
        y_true > 0.5,
        tf.float32,
    )
    y_pred = tf.cast(
        y_pred > SEG_THRESHOLD,
        tf.float32,
    )

    y_true_f = tf.reshape(
        y_true,
        [-1],
    )
    y_pred_f = tf.reshape(
        y_pred,
        [-1],
    )

    intersection = tf.reduce_sum(
        y_true_f * y_pred_f
    )

    union = (
        tf.reduce_sum(y_true_f)
        + tf.reduce_sum(y_pred_f)
        - intersection
    )

    return (
        intersection + EPS
    ) / (
        union + EPS
    )


@keras.utils.register_keras_serializable(
    package="MelanomaAI"
)
def combined_segmentation_loss(
    y_true,
    y_pred,
):
    bce = tf.reduce_mean(
        keras.losses.binary_crossentropy(
            y_true,
            y_pred,
        )
    )

    return (
        bce
        + dice_loss(y_true, y_pred)
    )


_test_true = tf.constant(
    [[[[1.0]], [[0.0]]]]
)
_test_pred = tf.constant(
    [[[[1.0]], [[0.0]]]]
)

print(
    "Perfect Dice:",
    float(
        dice_coefficient(
            _test_true,
            _test_pred,
        )
    ),
)

print(
    "Perfect IoU:",
    float(
        iou_coefficient(
            _test_true,
            _test_pred,
        )
    ),
)

## 12. Standard U-Net, Attention U-Net and ResUNet

All three use the same data split, image size, loss, optimiser, learning rate, batch size and epoch limit. ResUNet is tested because residual shortcuts may improve gradient flow while preserving U-Net localisation; superiority is not assumed in advance.


In [ ]:
def conv_block(x, filters, name):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv1")(x)
    x = layers.BatchNormalization(name=f"{name}_bn1")(x)
    x = layers.Activation("relu", name=f"{name}_relu1")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv2")(x)
    x = layers.BatchNormalization(name=f"{name}_bn2")(x)
    x = layers.Activation("relu", name=f"{name}_relu2")(x)
    return x

def attention_gate(skip, gating, inter_filters, name):
    theta = layers.Conv2D(inter_filters, 1, padding="same", name=f"{name}_theta")(skip)
    phi = layers.Conv2D(inter_filters, 1, padding="same", name=f"{name}_phi")(gating)
    x = layers.Add(name=f"{name}_add")([theta, phi])
    x = layers.Activation("relu", name=f"{name}_relu")(x)
    psi = layers.Conv2D(1, 1, padding="same", activation="sigmoid", name=f"{name}_psi")(x)
    return layers.Multiply(name=f"{name}_weighted_skip")([skip, psi])

def residual_block(x, filters, name):
    shortcut = x
    y = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv1")(x)
    y = layers.BatchNormalization(name=f"{name}_bn1")(y)
    y = layers.Activation("relu", name=f"{name}_relu1")(y)
    y = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv2")(y)
    y = layers.BatchNormalization(name=f"{name}_bn2")(y)

    if int(shortcut.shape[-1]) != filters:
        shortcut = layers.Conv2D(filters, 1, padding="same", use_bias=False, name=f"{name}_shortcut_conv")(shortcut)
        shortcut = layers.BatchNormalization(name=f"{name}_shortcut_bn")(shortcut)

    y = layers.Add(name=f"{name}_add")([y, shortcut])
    return layers.Activation("relu", name=f"{name}_out_relu")(y)

def build_standard_unet(input_shape=(256, 256, 3)):
    inputs = keras.Input(shape=input_shape, name="image")
    c1 = conv_block(inputs, 32, "enc1"); p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1, 64, "enc2"); p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2, 128, "enc3"); p3 = layers.MaxPooling2D()(c3)
    c4 = conv_block(p3, 256, "enc4"); p4 = layers.MaxPooling2D()(c4)
    b = conv_block(p4, 512, "bottleneck")

    def up(x, skip, filters, name):
        x = layers.Conv2DTranspose(filters, 2, strides=2, padding="same", name=f"{name}_up")(x)
        x = layers.Concatenate(name=f"{name}_concat")([x, skip])
        return conv_block(x, filters, name)

    d4 = up(b, c4, 256, "dec4")
    d3 = up(d4, c3, 128, "dec3")
    d2 = up(d3, c2, 64, "dec2")
    d1 = up(d2, c1, 32, "dec1")
    outputs = layers.Conv2D(1, 1, activation="sigmoid", dtype="float32", name="segmentation_mask")(d1)
    return keras.Model(inputs, outputs, name="Standard_U_Net")

def build_attention_unet(input_shape=(256, 256, 3)):
    inputs = keras.Input(shape=input_shape, name="image")
    c1 = conv_block(inputs, 32, "enc1"); p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1, 64, "enc2"); p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2, 128, "enc3"); p3 = layers.MaxPooling2D()(c3)
    c4 = conv_block(p3, 256, "enc4"); p4 = layers.MaxPooling2D()(c4)
    b = conv_block(p4, 512, "bottleneck")

    def up(x, skip, filters, level):
        x = layers.Conv2DTranspose(filters, 2, strides=2, padding="same", name=f"dec{level}_up")(x)
        skip = attention_gate(skip, x, max(filters // 2, 1), name=f"att{level}")
        x = layers.Concatenate(name=f"dec{level}_concat")([x, skip])
        return conv_block(x, filters, f"dec{level}")

    d4 = up(b, c4, 256, 4)
    d3 = up(d4, c3, 128, 3)
    d2 = up(d3, c2, 64, 2)
    d1 = up(d2, c1, 32, 1)
    outputs = layers.Conv2D(1, 1, activation="sigmoid", dtype="float32", name="segmentation_mask")(d1)
    return keras.Model(inputs, outputs, name="Attention_U_Net")

def build_resunet(input_shape=(256, 256, 3)):
    inputs = keras.Input(shape=input_shape, name="image")
    c1 = residual_block(inputs, 32, "enc1"); p1 = layers.MaxPooling2D()(c1)
    c2 = residual_block(p1, 64, "enc2"); p2 = layers.MaxPooling2D()(c2)
    c3 = residual_block(p2, 128, "enc3"); p3 = layers.MaxPooling2D()(c3)
    c4 = residual_block(p3, 256, "enc4"); p4 = layers.MaxPooling2D()(c4)
    b = residual_block(p4, 512, "bottleneck")

    def up(x, skip, filters, name):
        x = layers.Conv2DTranspose(filters, 2, strides=2, padding="same", name=f"{name}_up")(x)
        x = layers.Concatenate(name=f"{name}_concat")([x, skip])
        return residual_block(x, filters, name)

    d4 = up(b, c4, 256, "dec4")
    d3 = up(d4, c3, 128, "dec3")
    d2 = up(d3, c2, 64, "dec2")
    d1 = up(d2, c1, 32, "dec1")
    outputs = layers.Conv2D(1, 1, activation="sigmoid", dtype="float32", name="segmentation_mask")(d1)
    return keras.Model(inputs, outputs, name="ResUNet")

# Synthetic forward passes
dummy = tf.zeros((1, *SEG_IMAGE_SIZE, 3), dtype=tf.float32)
for builder in [build_standard_unet, build_attention_unet, build_resunet]:
    model = builder((*SEG_IMAGE_SIZE, 3))
    out = model(dummy, training=False)
    assert out.shape == (1, *SEG_IMAGE_SIZE, 1)
    print(model.name, "| params:", f"{model.count_params():,}", "| output:", out.shape)

# Explicit residual and attention tests
res_in = keras.Input(shape=(64, 64, 32))
res_test = keras.Model(res_in, residual_block(res_in, 64, "residual_test"))
assert res_test(tf.zeros((1, 64, 64, 32))).shape == (1, 64, 64, 64)

skip_in = keras.Input(shape=(64, 64, 64))
gate_in = keras.Input(shape=(64, 64, 64))
att_test = keras.Model([skip_in, gate_in], attention_gate(skip_in, gate_in, 32, "attention_test"))
assert att_test([tf.zeros((1, 64, 64, 64)), tf.zeros((1, 64, 64, 64))]).shape == (1, 64, 64, 64)

print("Residual block and attention gate self-tests passed.")

## 13. Fair training of all three segmentation models

In [ ]:
CUSTOM_OBJECTS = {
    "dice_coefficient": dice_coefficient,
    "dice_loss": dice_loss,
    "iou_coefficient": iou_coefficient,
    "combined_segmentation_loss": combined_segmentation_loss,
}

def compile_segmentation_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=SEG_LEARNING_RATE),
        loss=combined_segmentation_loss,
        metrics=[dice_coefficient, iou_coefficient],
    )

def segmentation_callbacks(model_path):
    return [
        keras.callbacks.ModelCheckpoint(
            model_path, monitor="val_dice_coefficient", mode="max",
            save_best_only=True, verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_dice_coefficient", mode="max",
            patience=6, restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_dice_coefficient", mode="max",
            factor=0.5, patience=3, min_lr=1e-6, verbose=1
        ),
    ]

SEGMENTATION_BUILDERS = {
    "Standard U-Net": build_standard_unet,
    "Attention U-Net": build_attention_unet,
    "ResUNet": build_resunet,
}

SEGMENTATION_PATHS = {
    "Standard U-Net": MODELS_DIR / "standard_unet_best.keras",
    "Attention U-Net": MODELS_DIR / "attention_unet_best.keras",
    "ResUNet": MODELS_DIR / "resunet_best.keras",
}

segmentation_runs = {}

for model_name, builder in SEGMENTATION_BUILDERS.items():
    print("\n" + "=" * 72)
    print("Training:", model_name)
    print("=" * 72)

    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model = builder((*SEG_IMAGE_SIZE, 3))
    compile_segmentation_model(model)

    start = time.time()
    history = model.fit(
        train_seg_ds,
        validation_data=val_seg_ds,
        epochs=SEG_EPOCHS,
        callbacks=segmentation_callbacks(SEGMENTATION_PATHS[model_name]),
        verbose=1,
    )
    elapsed = time.time() - start

    best_model = keras.models.load_model(
        SEGMENTATION_PATHS[model_name],
        custom_objects=CUSTOM_OBJECTS,
    )

    val_result = best_model.evaluate(val_seg_ds, verbose=0, return_dict=True)

    segmentation_runs[model_name] = {
        "history": history.history,
        "model_path": str(SEGMENTATION_PATHS[model_name]),
        "validation": val_result,
        "training_seconds": elapsed,
        "parameters": best_model.count_params(),
    }

    del model, best_model
    gc.collect()

print("Standard U-Net, Attention U-Net and ResUNet training complete.")

## 14. Validation comparison and objective segmenter selection

In [ ]:
rows = []
for name, run in segmentation_runs.items():
    rows.append({
        "Model": name,
        "Validation Dice": run["validation"]["dice_coefficient"],
        "Validation IoU": run["validation"]["iou_coefficient"],
        "Validation Loss": run["validation"]["loss"],
        "Parameter Count": run["parameters"],
        "Training Time (min)": run["training_seconds"] / 60.0,
    })

seg_comparison_df = pd.DataFrame(rows).sort_values(
    ["Validation Dice", "Validation IoU"],
    ascending=False,
).reset_index(drop=True)

display(seg_comparison_df.round(4))

SELECTED_SEGMENTER_NAME = seg_comparison_df.loc[0, "Model"]
SELECTED_SEGMENTER_PATH = Path(
    segmentation_runs[SELECTED_SEGMENTER_NAME]["model_path"]
)

selected_segmenter = keras.models.load_model(
    SELECTED_SEGMENTER_PATH,
    custom_objects=CUSTOM_OBJECTS,
)
selected_segmenter.save(MODELS_DIR / "selected_segmenter.keras")

print("Selected segmenter:", SELECTED_SEGMENTER_NAME)
print("Selection used validation data only.")

seg_comparison_df.to_csv(
    RESULTS_DIR / "segmentation_validation_comparison.csv",
    index=False,
)

## 15. Plot segmentation learning curves

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
)

for name, run in segmentation_runs.items():
    history = run["history"]

    epochs = range(
        1,
        len(history["loss"]) + 1,
    )

    axes[0].plot(
        epochs,
        history["loss"],
        label=f"{name} train",
    )

    axes[0].plot(
        epochs,
        history["val_loss"],
        linestyle="--",
        label=f"{name} val",
    )

    axes[1].plot(
        epochs,
        history[
            "dice_coefficient"
        ],
        label=f"{name} train",
    )

    axes[1].plot(
        epochs,
        history[
            "val_dice_coefficient"
        ],
        linestyle="--",
        label=f"{name} val",
    )

axes[0].set_title(
    "Segmentation Loss"
)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].set_title(
    "Segmentation Dice"
)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice")
axes[1].legend()

plt.tight_layout()
plt.show()

# PART B — LESION ROI GENERATION

The selected segmentation model is now locked.

The **predicted mask** is used to create the ROI supplied to EfficientNetB0 so the classification stage matches the intended end-to-end pipeline.

## 16. Predicted-mask and ROI extraction helpers

In [ ]:
def predict_mask_for_pil(
    model,
    image_pil,
):
    image_resized = image_pil.resize(
        (
            SEG_IMAGE_SIZE[1],
            SEG_IMAGE_SIZE[0],
        ),
        Image.Resampling.BILINEAR,
    )

    x = (
        np.asarray(image_resized)
        .astype("float32")
        / 255.0
    )

    prediction = model.predict(
        x[None, ...],
        verbose=0,
    )[0, ..., 0]

    binary = (
        prediction >= SEG_THRESHOLD
    ).astype(np.uint8)

    return prediction, binary


def extract_roi_from_binary_mask(
    image_pil,
    binary_mask,
    padding_fraction=0.10,
):
    """
    Extract a padded lesion crop.
    Falls back to the full image if
    the predicted mask is empty or invalid.
    """

    image_array = np.asarray(
        image_pil.convert("RGB")
    )

    original_h, original_w = (
        image_array.shape[:2]
    )

    mask_pil = Image.fromarray(
        (
            binary_mask * 255
        ).astype(np.uint8)
    )

    mask_full = mask_pil.resize(
        (
            original_w,
            original_h,
        ),
        Image.Resampling.NEAREST,
    )

    mask_full = (
        np.asarray(mask_full)
        > 0
    )

    ys, xs = np.where(
        mask_full
    )

    if (
        len(xs) == 0
        or len(ys) == 0
    ):
        return (
            image_pil.copy(),
            (
                0,
                0,
                original_w,
                original_h,
            ),
            True,
        )

    x_min = int(xs.min())
    x_max = int(xs.max())
    y_min = int(ys.min())
    y_max = int(ys.max())

    box_w = max(
        1,
        x_max - x_min + 1,
    )
    box_h = max(
        1,
        y_max - y_min + 1,
    )

    pad_x = int(
        round(
            box_w
            * padding_fraction
        )
    )

    pad_y = int(
        round(
            box_h
            * padding_fraction
        )
    )

    x0 = max(
        0,
        x_min - pad_x,
    )
    y0 = max(
        0,
        y_min - pad_y,
    )
    x1 = min(
        original_w,
        x_max + pad_x + 1,
    )
    y1 = min(
        original_h,
        y_max + pad_y + 1,
    )

    if (
        (x1 - x0) < 8
        or (y1 - y0) < 8
    ):
        return (
            image_pil.copy(),
            (
                0,
                0,
                original_w,
                original_h,
            ),
            True,
        )

    roi = image_pil.crop(
        (
            x0,
            y0,
            x1,
            y1,
        )
    )

    return (
        roi,
        (
            x0,
            y0,
            x1,
            y1,
        ),
        False,
    )


synthetic_image = Image.new(
    "RGB",
    (100, 80),
    "white",
)

empty_mask = np.zeros(
    (256, 256),
    dtype=np.uint8,
)

(
    roi_empty,
    box_empty,
    fallback_empty,
) = extract_roi_from_binary_mask(
    synthetic_image,
    empty_mask,
)

assert fallback_empty is True

normal_mask = np.zeros(
    (256, 256),
    dtype=np.uint8,
)
normal_mask[
    80:180,
    90:190,
] = 1

(
    roi_normal,
    box_normal,
    fallback_normal,
) = extract_roi_from_binary_mask(
    synthetic_image,
    normal_mask,
)

assert fallback_normal is False

print("ROI safety tests passed.")

## 17. Generate ROI files for all selected cases

In [ ]:
ROI_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

roi_records = []

for index, row in selected_df.iterrows():
    image_pil = load_rgb(
        row["image_path"]
    )

    (
        probability_mask,
        binary_mask,
    ) = predict_mask_for_pil(
        selected_segmenter,
        image_pil,
    )

    (
        roi,
        bbox,
        fallback,
    ) = extract_roi_from_binary_mask(
        image_pil,
        binary_mask,
        padding_fraction=0.10,
    )

    roi_path = (
        ROI_DIR
        / f"{row['image_id']}_ROI.jpg"
    )

    roi.convert("RGB").save(
        roi_path,
        quality=95,
    )

    roi_records.append(
        {
            "image_id": row["image_id"],
            "roi_path": str(roi_path),
            "roi_fallback": bool(
                fallback
            ),
            "bbox": json.dumps(
                [
                    int(value)
                    for value
                    in bbox
                ]
            ),
        }
    )

    if (
        index + 1
    ) % 25 == 0:
        print(
            f"Generated "
            f"{index + 1}/200 ROIs"
        )

roi_df = pd.DataFrame(
    roi_records
)


def attach_rois(frame):
    result = frame.merge(
        roi_df,
        on="image_id",
        how="left",
    )

    assert (
        result["roi_path"]
        .notna()
        .all()
    )

    return result


train_df = attach_rois(
    train_df
)
val_df = attach_rois(
    val_df
)
model_df = attach_rois(
    model_df
)
test_df = attach_rois(
    test_df
)

print("ROI generation complete.")
print(
    "Fallback ROIs:",
    int(
        roi_df[
            "roi_fallback"
        ].sum()
    ),
    "out of 200",
)

## 18. Visualise original → predicted mask → ROI

In [ ]:
example = test_df.iloc[0]

image_pil = load_rgb(
    example["image_path"]
)

(
    probability_mask,
    binary_mask,
) = predict_mask_for_pil(
    selected_segmenter,
    image_pil,
)

roi_pil = load_rgb(
    example["roi_path"]
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(14, 4),
)

axes[0].imshow(
    image_pil
)
axes[0].set_title(
    "Original"
)
axes[0].axis(
    "off"
)

axes[1].imshow(
    binary_mask,
    cmap="gray",
)
axes[1].set_title(
    "Predicted lesion mask"
)
axes[1].axis(
    "off"
)

axes[2].imshow(
    roi_pil
)
axes[2].set_title(
    "Lesion ROI"
)
axes[2].axis(
    "off"
)

plt.tight_layout()
plt.show()

# PART C — EFFICIENTNETB0 CLASSIFICATION

EfficientNetB0 performs binary classification:

- `0 = benign`
- `1 = malignant`

Transfer learning is used because the modelling dataset is small.

## 19. `tf.data` classification pipeline

In [ ]:
classification_augmentation = (
    keras.Sequential(
        [
            layers.RandomFlip(
                "horizontal_and_vertical",
                seed=SEED,
            ),
            layers.RandomRotation(
                0.05,
                seed=SEED,
            ),
            layers.RandomZoom(
                0.10,
                seed=SEED,
            ),
            layers.RandomContrast(
                0.10,
                seed=SEED,
            ),
        ],
        name=(
            "classification_augmentation"
        ),
    )
)


def decode_classification_image(
    path,
    label,
):
    image_bytes = tf.io.read_file(
        path
    )

    image = tf.io.decode_jpeg(
        image_bytes,
        channels=3,
    )

    image = tf.image.resize(
        image,
        CLS_IMAGE_SIZE,
        method="bilinear",
    )

    # EfficientNetB0 in Keras contains
    # its own input Rescaling layer.
    # Keep pixels in the 0-255 range.
    image = tf.cast(
        image,
        tf.float32,
    )

    label = tf.cast(
        label,
        tf.float32,
    )

    return image, label


def make_cls_dataset(
    frame,
    training=False,
):
    ds = (
        tf.data.Dataset
        .from_tensor_slices(
            (
                frame[
                    "roi_path"
                ]
                .astype(str)
                .values,
                frame[
                    "label"
                ]
                .astype(
                    np.float32
                )
                .values,
            )
        )
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    ds = ds.map(
        decode_classification_image,
        num_parallel_calls=AUTOTUNE,
    )

    ds = ds.batch(
        CLS_BATCH_SIZE
    )

    ds = ds.prefetch(
        AUTOTUNE
    )

    return ds


train_cls_ds = make_cls_dataset(
    train_df,
    training=True,
)

val_cls_ds = make_cls_dataset(
    val_df,
    training=False,
)

test_cls_ds = make_cls_dataset(
    test_df,
    training=False,
)

cls_images, cls_labels = next(
    iter(train_cls_ds)
)

print(
    "Classification image batch:",
    cls_images.shape,
)
print(
    "Pixel range:",
    float(
        tf.reduce_min(
            cls_images
        )
    ),
    "to",
    float(
        tf.reduce_max(
            cls_images
        )
    ),
)
print(
    "Labels:",
    cls_labels.numpy()[:10],
)

## 20. Class weights

In [ ]:
train_labels = (
    train_df["label"]
    .astype(int)
    .values
)

classes = np.unique(
    train_labels
)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels,
)

CLASS_WEIGHTS = {
    int(class_value): float(weight)
    for class_value, weight
    in zip(classes, weights)
}

print(
    "Training class counts:"
)

display(
    train_df[
        "diagnosis"
    ]
    .value_counts()
    .to_frame("count")
)

print(
    "Class weights:",
    CLASS_WEIGHTS,
)

## 21. Build EfficientNetB0 transfer-learning classifier

In [ ]:
def build_efficientnet_classifier():
    inputs = keras.Input(
        shape=(
            *CLS_IMAGE_SIZE,
            3,
        ),
        name="roi_image",
    )

    x = (
        classification_augmentation(
            inputs
        )
    )

    backbone = (
        keras.applications
        .EfficientNetB0(
            include_top=False,
            weights="imagenet",
            input_shape=(
                *CLS_IMAGE_SIZE,
                3,
            ),
        )
    )

    backbone._name = (
        "efficientnetb0"
    )

    backbone.trainable = False

    x = backbone(
        x,
        training=False,
    )

    x = layers.GlobalAveragePooling2D(
        name="gap"
    )(x)

    x = layers.Dropout(
        0.30,
        name="dropout",
    )(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="prediction",
    )(x)

    return keras.Model(
        inputs,
        outputs,
        name=(
            "Melanoma_EfficientNetB0"
        ),
    )


classifier = (
    build_efficientnet_classifier()
)

classifier.compile(
    optimizer=keras.optimizers.Adam(
        CLS_HEAD_LR
    ),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),
        keras.metrics.AUC(
            name="auc"
        ),
        keras.metrics.Precision(
            name="precision"
        ),
        keras.metrics.Recall(
            name="recall"
        ),
    ],
)

classifier.summary()

dummy_cls = tf.zeros(
    (
        1,
        224,
        224,
        3,
    ),
    dtype=tf.float32,
)

dummy_output = classifier(
    dummy_cls,
    training=False,
)

assert dummy_output.shape == (
    1,
    1,
)

print(
    "EfficientNetB0 classifier "
    "passed the synthetic shape test."
)

## 22. Train classification head

In [ ]:
HEAD_MODEL_PATH = (
    MODELS_DIR
    / "efficientnet_b0_head_best.keras"
)

head_callbacks = [
    keras.callbacks.ModelCheckpoint(
        HEAD_MODEL_PATH,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

head_history = classifier.fit(
    train_cls_ds,
    validation_data=val_cls_ds,
    epochs=CLS_HEAD_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=head_callbacks,
    verbose=1,
)

## 23. Optional fine-tuning

In [ ]:
FINE_MODEL_PATH = (
    MODELS_DIR
    / "efficientnet_b0_finetuned_best.keras"
)

fine_history = None

if DO_FINE_TUNING:
    classifier = keras.models.load_model(
        HEAD_MODEL_PATH
    )

    backbone = classifier.get_layer(
        "efficientnetb0"
    )

    backbone.trainable = True

    for layer in backbone.layers[:-20]:
        layer.trainable = False

    for layer in backbone.layers:
        if isinstance(
            layer,
            layers.BatchNormalization,
        ):
            layer.trainable = False

    classifier.compile(
        optimizer=keras.optimizers.Adam(
            CLS_FINE_TUNE_LR
        ),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(
                name="accuracy"
            ),
            keras.metrics.AUC(
                name="auc"
            ),
            keras.metrics.Precision(
                name="precision"
            ),
            keras.metrics.Recall(
                name="recall"
            ),
        ],
    )

    fine_callbacks = [
        keras.callbacks.ModelCheckpoint(
            FINE_MODEL_PATH,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=4,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc",
            mode="max",
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1,
        ),
    ]

    fine_history = classifier.fit(
        train_cls_ds,
        validation_data=val_cls_ds,
        epochs=CLS_FINE_TUNE_EPOCHS,
        class_weight=CLASS_WEIGHTS,
        callbacks=fine_callbacks,
        verbose=1,
    )
else:
    print(
        "Fine-tuning disabled."
    )

## Optional classifier baseline — MobileNetV2

In [ ]:
baseline_validation_row = None

if RUN_CLASSIFIER_BASELINE:
    def build_mobilenetv2_baseline():
        inputs = keras.Input(shape=(*CLS_IMAGE_SIZE, 3), name="roi_image")
        x = classification_augmentation(inputs)
        x = keras.applications.mobilenet_v2.preprocess_input(x)

        backbone = keras.applications.MobileNetV2(
            include_top=False,
            weights="imagenet",
            input_shape=(*CLS_IMAGE_SIZE, 3),
            name="mobilenetv2",
        )
        backbone.trainable = False

        x = backbone(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(0.30)(x)
        outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
        return keras.Model(inputs, outputs, name="MobileNetV2_Baseline")

    baseline = build_mobilenetv2_baseline()
    baseline.compile(
        optimizer=keras.optimizers.Adam(CLS_HEAD_LR),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")],
    )

    baseline_path = MODELS_DIR / "mobilenetv2_baseline_best.keras"

    baseline.fit(
        train_cls_ds,
        validation_data=val_cls_ds,
        epochs=CLS_HEAD_EPOCHS,
        class_weight=CLASS_WEIGHTS,
        callbacks=[
            keras.callbacks.ModelCheckpoint(
                baseline_path, monitor="val_auc", mode="max",
                save_best_only=True, verbose=1
            ),
            keras.callbacks.EarlyStopping(
                monitor="val_auc", mode="max",
                patience=5, restore_best_weights=True
            ),
        ],
        verbose=1,
    )

    baseline = keras.models.load_model(baseline_path)
    baseline_val_probs = baseline.predict(val_cls_ds, verbose=0).reshape(-1)
    val_true_for_baseline = val_df["label"].astype(int).values
    baseline_pred = (baseline_val_probs >= 0.5).astype(int)
    cm_b = confusion_matrix(val_true_for_baseline, baseline_pred, labels=[0,1])
    tn_b, fp_b, fn_b, tp_b = cm_b.ravel()

    baseline_validation_row = {
        "Model": "MobileNetV2 baseline",
        "Path": str(baseline_path),
        "Validation ROC-AUC": roc_auc_score(val_true_for_baseline, baseline_val_probs),
        "Validation Sensitivity": tp_b/(tp_b+fn_b) if (tp_b+fn_b) else np.nan,
        "Validation Specificity": tn_b/(tn_b+fp_b) if (tn_b+fp_b) else np.nan,
        "Validation F1": f1_score(val_true_for_baseline, baseline_pred, zero_division=0),
    }

    print("MobileNetV2 validation comparison:")
    display(pd.DataFrame([baseline_validation_row]).round(4))
else:
    print("RUN_CLASSIFIER_BASELINE=False — optional MobileNetV2 baseline skipped.")

## 24. Select the classifier using validation ROC-AUC

In [ ]:
def predict_probabilities(model, dataset):
    return model.predict(dataset, verbose=0).reshape(-1)

def validation_metrics(y_true, probs, threshold=0.5):
    pred = (probs >= threshold).astype(int)
    cm_v = confusion_matrix(y_true, pred, labels=[0,1])
    tn_v, fp_v, fn_v, tp_v = cm_v.ravel()
    return {
        "Validation Sensitivity": tp_v/(tp_v+fn_v) if (tp_v+fn_v) else np.nan,
        "Validation Specificity": tn_v/(tn_v+fp_v) if (tn_v+fp_v) else np.nan,
        "Validation F1": f1_score(y_true, pred, zero_division=0),
    }

val_true = val_df["label"].astype(int).values
candidate_rows = []

head_model = keras.models.load_model(HEAD_MODEL_PATH)
head_probs = predict_probabilities(head_model, val_cls_ds)
m = validation_metrics(val_true, head_probs)
candidate_rows.append({
    "Model": "Frozen EfficientNetB0 head",
    "Path": str(HEAD_MODEL_PATH),
    "Validation ROC-AUC": roc_auc_score(val_true, head_probs),
    **m,
})

if DO_FINE_TUNING and FINE_MODEL_PATH.exists():
    fine_model = keras.models.load_model(FINE_MODEL_PATH)
    fine_probs = predict_probabilities(fine_model, val_cls_ds)
    m = validation_metrics(val_true, fine_probs)
    candidate_rows.append({
        "Model": "Fine-tuned EfficientNetB0",
        "Path": str(FINE_MODEL_PATH),
        "Validation ROC-AUC": roc_auc_score(val_true, fine_probs),
        **m,
    })

if baseline_validation_row is not None:
    candidate_rows.append(baseline_validation_row)

classifier_comparison_df = pd.DataFrame(candidate_rows).sort_values(
    "Validation ROC-AUC", ascending=False
).reset_index(drop=True)

display(classifier_comparison_df.round(4))

SELECTED_CLASSIFIER_NAME = classifier_comparison_df.loc[0, "Model"]
SELECTED_CLASSIFIER_PATH = Path(classifier_comparison_df.loc[0, "Path"])

final_classifier = keras.models.load_model(SELECTED_CLASSIFIER_PATH)
final_classifier.save(MODELS_DIR / "selected_classifier.keras")

selected_val_probs = predict_probabilities(final_classifier, val_cls_ds)

if OPTIMISE_THRESHOLD_ON_VALIDATION and len(np.unique(val_true)) == 2:
    fpr_v, tpr_v, thresholds_v = roc_curve(val_true, selected_val_probs)
    youden_j = tpr_v - fpr_v
    finite = np.isfinite(thresholds_v)
    best_idx = np.argmax(np.where(finite, youden_j, -np.inf))
    SELECTED_CLS_THRESHOLD = float(np.clip(thresholds_v[best_idx], 0.0, 1.0))
else:
    SELECTED_CLS_THRESHOLD = DEFAULT_CLS_THRESHOLD

threshold_info = {
    "default_threshold": DEFAULT_CLS_THRESHOLD,
    "selected_validation_threshold": SELECTED_CLS_THRESHOLD,
    "method": "maximum Youden J on validation probabilities"
    if OPTIMISE_THRESHOLD_ON_VALIDATION else "default 0.5",
}

(RESULTS_DIR / "classification_threshold.json").write_text(
    json.dumps(threshold_info, indent=2),
    encoding="utf-8",
)

classifier_comparison_df.to_csv(
    RESULTS_DIR / "classifier_validation_comparison.csv",
    index=False,
)

print("Selected classifier:", SELECTED_CLASSIFIER_NAME)
print(json.dumps(threshold_info, indent=2))

## 25. Classification learning curves

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5),
)

head_epochs = range(
    1,
    len(
        head_history.history[
            "loss"
        ]
    ) + 1,
)

axes[0].plot(
    head_epochs,
    head_history.history[
        "loss"
    ],
    label="Head train",
)

axes[0].plot(
    head_epochs,
    head_history.history[
        "val_loss"
    ],
    label="Head val",
)

axes[1].plot(
    head_epochs,
    head_history.history[
        "auc"
    ],
    label="Head train",
)

axes[1].plot(
    head_epochs,
    head_history.history[
        "val_auc"
    ],
    label="Head val",
)

if fine_history is not None:
    offset = len(
        head_epochs
    )

    fine_epochs = range(
        offset + 1,
        offset
        + len(
            fine_history.history[
                "loss"
            ]
        )
        + 1,
    )

    axes[0].plot(
        fine_epochs,
        fine_history.history[
            "loss"
        ],
        label="Fine train",
    )

    axes[0].plot(
        fine_epochs,
        fine_history.history[
            "val_loss"
        ],
        label="Fine val",
    )

    axes[1].plot(
        fine_epochs,
        fine_history.history[
            "auc"
        ],
        label="Fine train",
    )

    axes[1].plot(
        fine_epochs,
        fine_history.history[
            "val_auc"
        ],
        label="Fine val",
    )

axes[0].set_title(
    "Classification loss"
)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].set_title(
    "Classification ROC-AUC"
)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("ROC-AUC")
axes[1].legend()

plt.tight_layout()
plt.show()

# PART D — FINAL HELD-OUT TESTING

Only now are the 100 held-out test cases used for final performance reporting.

Final classification results include:

- True Positive (TP)
- True Negative (TN)
- False Positive (FP)
- False Negative (FN)
- sensitivity
- specificity
- accuracy
- balanced accuracy
- precision
- F1-score
- ROC-AUC

## 26. Final segmentation testing on the 100 held-out cases

In [ ]:
def evaluate_segmentation_on_dataframe(
    model,
    frame,
):
    dice_values = []
    iou_values = []

    tp_pixels = 0
    tn_pixels = 0
    fp_pixels = 0
    fn_pixels = 0

    for _, row in frame.iterrows():
        image_pil = load_rgb(
            row["image_path"]
        )

        true_mask_pil = load_mask(
            row["mask_path"]
        )

        true_mask = true_mask_pil.resize(
            (
                SEG_IMAGE_SIZE[1],
                SEG_IMAGE_SIZE[0],
            ),
            Image.Resampling.NEAREST,
        )

        true_mask = (
            np.asarray(true_mask)
            > 127
        ).astype(np.uint8)

        _, pred_mask = predict_mask_for_pil(
            model,
            image_pil,
        )

        intersection = np.logical_and(
            true_mask == 1,
            pred_mask == 1,
        ).sum()

        true_sum = (
            true_mask == 1
        ).sum()

        pred_sum = (
            pred_mask == 1
        ).sum()

        union = np.logical_or(
            true_mask == 1,
            pred_mask == 1,
        ).sum()

        dice = (
            2 * intersection
            + EPS
        ) / (
            true_sum
            + pred_sum
            + EPS
        )

        iou = (
            intersection
            + EPS
        ) / (
            union
            + EPS
        )

        dice_values.append(
            dice
        )

        iou_values.append(
            iou
        )

        tp_pixels += np.logical_and(
            true_mask == 1,
            pred_mask == 1,
        ).sum()

        tn_pixels += np.logical_and(
            true_mask == 0,
            pred_mask == 0,
        ).sum()

        fp_pixels += np.logical_and(
            true_mask == 0,
            pred_mask == 1,
        ).sum()

        fn_pixels += np.logical_and(
            true_mask == 1,
            pred_mask == 0,
        ).sum()

    sensitivity = (
        tp_pixels
        / (
            tp_pixels
            + fn_pixels
            + EPS
        )
    )

    specificity = (
        tn_pixels
        / (
            tn_pixels
            + fp_pixels
            + EPS
        )
    )

    return {
        "Mean Dice": float(
            np.mean(dice_values)
        ),
        "Mean IoU": float(
            np.mean(iou_values)
        ),
        "Pixel sensitivity": float(
            sensitivity
        ),
        "Pixel specificity": float(
            specificity
        ),
    }


seg_test_rows = []

for model_name in SEGMENTATION_BUILDERS:
    model = (
        keras.models.load_model(
            SEGMENTATION_PATHS[model_name],
            custom_objects=(
                CUSTOM_OBJECTS
            ),
        )
    )

    metrics = (
        evaluate_segmentation_on_dataframe(
            model,
            test_df,
        )
    )

    metrics["Model"] = (
        model_name
    )

    seg_test_rows.append(
        metrics
    )

seg_test_df = pd.DataFrame(
    seg_test_rows
)

display(
    seg_test_df.round(4)
)

seg_test_df.to_csv(
    RESULTS_DIR
    / "segmentation_final_test_results.csv",
    index=False,
)

# Merge validation + test metrics into one model-comparison table.
combined_rows = []
for _, row in seg_test_df.iterrows():
    model_name = row["Model"]
    val_row = seg_comparison_df[seg_comparison_df["Model"] == model_name].iloc[0]
    combined_rows.append({
        "Model": model_name,
        "Validation Dice": val_row["Validation Dice"],
        "Validation IoU": val_row["Validation IoU"],
        "Test Dice": row["Mean Dice"],
        "Test IoU": row["Mean IoU"],
        "Pixel Sensitivity": row["Pixel sensitivity"],
        "Pixel Specificity": row["Pixel specificity"],
        "Parameter Count": val_row["Parameter Count"],
        "Training Time (min)": val_row["Training Time (min)"],
    })

segmentation_model_comparison_df = pd.DataFrame(combined_rows)

unetpp_placeholder = pd.DataFrame([{
    "Model": "U-Net++ (Daisy — enter real team results)",
    "Validation Dice": np.nan,
    "Validation IoU": np.nan,
    "Test Dice": np.nan,
    "Test IoU": np.nan,
    "Pixel Sensitivity": np.nan,
    "Pixel Specificity": np.nan,
    "Parameter Count": np.nan,
    "Training Time (min)": np.nan,
}])

team_segmentation_comparison_df = pd.concat(
    [segmentation_model_comparison_df, unetpp_placeholder],
    ignore_index=True,
)

display(team_segmentation_comparison_df.round(4))

segmentation_model_comparison_df.to_csv(
    RESULTS_DIR / "segmentation_model_comparison.csv",
    index=False,
)
team_segmentation_comparison_df.to_csv(
    RESULTS_DIR / "segmentation_team_comparison_with_unetpp_placeholder.csv",
    index=False,
)


## Qualitative segmentation comparison

The figures below follow the same presentation style as Daisy's U-Net++ example:
**Image → Ground Truth → Prediction → Overlay**.

In [ ]:
def qualitative_overlay(image_pil, pred_mask, alpha=0.35):
    resized = image_pil.resize(
        (SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]),
        Image.Resampling.BILINEAR,
    )
    base = np.asarray(resized).astype(np.float32) / 255.0
    overlay = base.copy()
    mask = pred_mask.astype(bool)
    overlay[mask, 0] = np.clip(overlay[mask, 0] * (1 - alpha) + alpha, 0, 1)
    overlay[mask, 1] *= (1 - alpha)
    overlay[mask, 2] *= (1 - alpha)
    return overlay

def true_mask_array(mask_path):
    m = load_mask(mask_path).resize(
        (SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]),
        Image.Resampling.NEAREST,
    )
    return (np.asarray(m) > 127).astype(np.uint8)

def show_model_samples(model_name):
    model = keras.models.load_model(
        SEGMENTATION_PATHS[model_name],
        custom_objects=CUSTOM_OBJECTS,
    )

    samples = pd.concat([
        test_df[test_df["label"] == 0].head(1),
        test_df[test_df["label"] == 1].head(1),
    ], ignore_index=True)

    fig, axes = plt.subplots(len(samples), 4, figsize=(14, 3.5 * len(samples)))
    if len(samples) == 1:
        axes = np.array([axes])

    for i, row in samples.iterrows():
        image = load_rgb(row["image_path"])
        gt = true_mask_array(row["mask_path"])
        _, pred = predict_mask_for_pil(model, image)

        intersection = np.logical_and(gt == 1, pred == 1).sum()
        true_sum = (gt == 1).sum()
        pred_sum = (pred == 1).sum()
        union = np.logical_or(gt == 1, pred == 1).sum()

        dice = (2 * intersection + EPS) / (true_sum + pred_sum + EPS)
        iou = (intersection + EPS) / (union + EPS)

        axes[i, 0].imshow(image)
        axes[i, 0].set_title(f"Image\nTrue class: {row['diagnosis'].title()}")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(gt, cmap="gray")
        axes[i, 1].set_title("Ground Truth")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(pred, cmap="gray")
        axes[i, 2].set_title(f"Prediction\nDice={dice:.3f}, IoU={iou:.3f}")
        axes[i, 2].axis("off")

        axes[i, 3].imshow(qualitative_overlay(image, pred))
        axes[i, 3].set_title("Overlay")
        axes[i, 3].axis("off")

    fig.suptitle(f"{model_name} — Final Test Samples", fontsize=15)
    plt.tight_layout()
    plt.show()

    del model
    gc.collect()

for name in SEGMENTATION_BUILDERS:
    show_model_samples(name)

## Direct same-lesion comparison: Standard U-Net vs Attention U-Net vs ResUNet

In [ ]:
def direct_comparison(row):
    image = load_rgb(row["image_path"])
    gt = true_mask_array(row["mask_path"])

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    axes[0].imshow(image)
    axes[0].set_title(f"Original\n{row['diagnosis'].title()}")
    axes[0].axis("off")

    axes[1].imshow(gt, cmap="gray")
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")

    for ax_i, model_name in enumerate(SEGMENTATION_BUILDERS, start=2):
        model = keras.models.load_model(
            SEGMENTATION_PATHS[model_name],
            custom_objects=CUSTOM_OBJECTS,
        )
        _, pred = predict_mask_for_pil(model, image)

        inter = np.logical_and(gt == 1, pred == 1).sum()
        true_sum = (gt == 1).sum()
        pred_sum = (pred == 1).sum()
        union = np.logical_or(gt == 1, pred == 1).sum()

        dice = (2 * inter + EPS) / (true_sum + pred_sum + EPS)
        iou = (inter + EPS) / (union + EPS)

        axes[ax_i].imshow(pred, cmap="gray")
        axes[ax_i].set_title(f"{model_name}\nDice={dice:.3f}\nIoU={iou:.3f}")
        axes[ax_i].axis("off")
        del model

    plt.tight_layout()
    plt.show()
    gc.collect()

direct_comparison(test_df[test_df["label"] == 0].iloc[0])
direct_comparison(test_df[test_df["label"] == 1].iloc[0])

## 27. Final classification confusion matrix and clinical metrics

In [ ]:
test_true = test_df["label"].astype(int).values
test_probs = final_classifier.predict(test_cls_ds, verbose=0).reshape(-1)

def metrics_at_threshold(y_true, probs, threshold):
    pred = (probs >= threshold).astype(int)
    cm_local = confusion_matrix(y_true, pred, labels=[0,1])
    tn_l, fp_l, fn_l, tp_l = cm_local.ravel()

    sensitivity_l = tp_l/(tp_l+fn_l) if (tp_l+fn_l) else np.nan
    specificity_l = tn_l/(tn_l+fp_l) if (tn_l+fp_l) else np.nan

    return {
        "Threshold": float(threshold),
        "TP": int(tp_l),
        "TN": int(tn_l),
        "FP": int(fp_l),
        "FN": int(fn_l),
        "Sensitivity": float(sensitivity_l),
        "Specificity": float(specificity_l),
        "Accuracy": float(accuracy_score(y_true, pred)),
        "Balanced Accuracy": float(balanced_accuracy_score(y_true, pred)),
        "Precision": float(precision_score(y_true, pred, zero_division=0)),
        "F1": float(f1_score(y_true, pred, zero_division=0)),
        "ROC-AUC": float(roc_auc_score(y_true, probs)) if len(np.unique(y_true)) == 2 else np.nan,
        "PR-AUC": float(average_precision_score(y_true, probs)) if len(np.unique(y_true)) == 2 else np.nan,
    }, cm_local, pred

default_metrics, default_cm, default_pred = metrics_at_threshold(
    test_true, test_probs, DEFAULT_CLS_THRESHOLD
)
selected_metrics, cm, test_pred = metrics_at_threshold(
    test_true, test_probs, SELECTED_CLS_THRESHOLD
)

final_metrics = pd.DataFrame([
    {"Evaluation": "Default threshold 0.5", **default_metrics},
    {"Evaluation": "Validation-selected threshold", **selected_metrics},
])

display(final_metrics.round(4))

print("\nClassification report at validation-selected threshold:")
print(classification_report(
    test_true, test_pred,
    target_names=["Benign", "Malignant"],
    zero_division=0,
))

final_metrics.to_csv(RESULTS_DIR / "classification_metrics.csv", index=False)

tn, fp, fn, tp = cm.ravel()
sensitivity = selected_metrics["Sensitivity"]
specificity = selected_metrics["Specificity"]
accuracy = selected_metrics["Accuracy"]
roc_auc = selected_metrics["ROC-AUC"]

test_predictions_df = test_df[
    ["image_id", "diagnosis", "label", "image_path", "mask_path", "roi_path"]
].copy()
test_predictions_df["malignancy_probability"] = test_probs
test_predictions_df["predicted_label"] = test_pred
test_predictions_df["predicted_diagnosis"] = np.where(test_pred == 1, "malignant", "benign")
test_predictions_df.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

## 28. Confusion matrix visualisation

In [ ]:
fig, ax = plt.subplots(
    figsize=(6, 5)
)

ax.imshow(
    cm
)

ax.set_xticks(
    [0, 1]
)
ax.set_yticks(
    [0, 1]
)

ax.set_xticklabels(
    [
        "Predicted Benign",
        "Predicted Melanoma",
    ]
)

ax.set_yticklabels(
    [
        "Actual Benign",
        "Actual Melanoma",
    ]
)

for i in range(2):
    for j in range(2):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            fontsize=18,
        )

ax.set_title(
    f"Final Test Confusion Matrix (n={len(test_df)})\nThreshold={SELECTED_CLS_THRESHOLD:.3f}"
)
ax.set_xlabel(
    "Model prediction"
)
ax.set_ylabel(
    "Ground truth"
)

plt.tight_layout()
plt.show()

print(f"TN = {tn}")
print(f"FP = {fp}")
print(f"FN = {fn}")
print(f"TP = {tp}")

print(
    f"Sensitivity = "
    f"{sensitivity:.3f}"
)
print(
    f"Specificity = "
    f"{specificity:.3f}"
)
print(
    f"Accuracy = "
    f"{accuracy:.3f}"
)

## 29. ROC curve

In [ ]:
if len(np.unique(test_true)) == 2:
    fpr, tpr, _ = roc_curve(test_true, test_probs)
    precision_curve, recall_curve, _ = precision_recall_curve(test_true, test_probs)

    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f"ROC-AUC={selected_metrics['ROC-AUC']:.3f}")
    plt.plot([0,1],[0,1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate / Sensitivity")
    plt.title("Final Test ROC Curve")
    plt.legend()
    plt.show()

    plt.figure(figsize=(6,5))
    plt.plot(recall_curve, precision_curve, label=f"PR-AUC={selected_metrics['PR-AUC']:.3f}")
    plt.xlabel("Recall / Sensitivity")
    plt.ylabel("Precision")
    plt.title("Final Test Precision–Recall Curve")
    plt.legend()
    plt.show()
else:
    print("ROC/PR curves require both classes.")

## 30. How to interpret the confusion matrix

- **True Positive (TP):** actual melanoma correctly predicted as melanoma.
- **True Negative (TN):** actual benign lesion correctly predicted as benign.
- **False Positive (FP):** benign lesion incorrectly predicted as melanoma.
- **False Negative (FN):** melanoma incorrectly predicted as benign.

### Sensitivity

\[
Sensitivity = \frac{TP}{TP + FN}
\]

Sensitivity asks:

> Of all true melanoma cases, how many did the AI correctly detect?

### Specificity

\[
Specificity = \frac{TN}{TN + FP}
\]

Specificity asks:

> Of all true benign cases, how many did the AI correctly identify as benign?

False negatives are particularly important in melanoma screening because they represent malignant lesions the model failed to flag.

## 31. Display example TP / TN / FP / FN predictions

In [ ]:
def outcome_type(
    true_label,
    pred_label,
):
    if (
        true_label == 1
        and pred_label == 1
    ):
        return "TP"

    if (
        true_label == 0
        and pred_label == 0
    ):
        return "TN"

    if (
        true_label == 0
        and pred_label == 1
    ):
        return "FP"

    return "FN"


test_predictions_df[
    "outcome"
] = [
    outcome_type(
        true_label,
        pred_label,
    )
    for true_label, pred_label
    in zip(
        test_true,
        test_pred,
    )
]

examples = []

for outcome in [
    "TP",
    "TN",
    "FP",
    "FN",
]:
    subset = (
        test_predictions_df[
            test_predictions_df[
                "outcome"
            ] == outcome
        ]
    )

    if len(subset):
        examples.append(
            subset.iloc[0]
        )

if examples:
    fig, axes = plt.subplots(
        len(examples),
        2,
        figsize=(
            10,
            4 * len(examples),
        ),
    )

    if len(examples) == 1:
        axes = np.array(
            [axes]
        )

    for row_idx, row in enumerate(
        examples
    ):
        original = load_rgb(
            row["image_path"]
        )

        roi = load_rgb(
            row["roi_path"]
        )

        axes[
            row_idx,
            0,
        ].imshow(
            original
        )

        axes[
            row_idx,
            0,
        ].set_title(
            f"{row['outcome']} | "
            f"True: {row['diagnosis']}"
        )

        axes[
            row_idx,
            0,
        ].axis(
            "off"
        )

        axes[
            row_idx,
            1,
        ].imshow(
            roi
        )

        axes[
            row_idx,
            1,
        ].set_title(
            f"Pred: "
            f"{row['predicted_diagnosis']} | "
            f"P(malignant)="
            f"{row['malignancy_probability']:.3f}"
        )

        axes[
            row_idx,
            1,
        ].axis(
            "off"
        )

    plt.tight_layout()
    plt.show()
else:
    print(
        "No example outcomes available."
    )

## Dedicated false-negative analysis

In [ ]:
false_negatives = test_predictions_df[
    test_predictions_df["outcome"] == "FN"
].copy()

print("False negatives:", len(false_negatives))

if len(false_negatives):
    display(
        false_negatives[
            ["image_id", "diagnosis", "malignancy_probability", "predicted_diagnosis"]
        ].sort_values("malignancy_probability")
    )
else:
    print("No false negatives occurred at the selected threshold in this run.")

# PART E — GRAD-CAM EXPLAINABILITY

Grad-CAM visualises spatial regions that most influenced the classifier output.

It does **not** prove clinical causality and should not be interpreted as a diagnostic explanation.

## 32. Grad-CAM helper

In [ ]:
def make_gradcam_heatmap(
    model,
    image_batch,
):
    backbone = model.get_layer(
        "efficientnetb0"
    )

    gap = model.get_layer(
        "gap"
    )

    dropout = model.get_layer(
        "dropout"
    )

    prediction_layer = model.get_layer(
        "prediction"
    )

    with tf.GradientTape() as tape:
        feature_maps = backbone(
            image_batch,
            training=False,
        )

        tape.watch(
            feature_maps
        )

        x = gap(
            feature_maps
        )

        x = dropout(
            x,
            training=False,
        )

        prediction = (
            prediction_layer(
                x
            )
        )

    gradients = tape.gradient(
        prediction,
        feature_maps,
    )

    pooled_gradients = (
        tf.reduce_mean(
            gradients,
            axis=(1, 2),
        )
    )

    feature_maps = (
        feature_maps[0]
    )

    pooled_gradients = (
        pooled_gradients[0]
    )

    heatmap = tf.reduce_sum(
        feature_maps
        * pooled_gradients,
        axis=-1,
    )

    heatmap = tf.maximum(
        heatmap,
        0,
    )

    max_value = tf.reduce_max(
        heatmap
    )

    heatmap = (
        heatmap
        / (
            max_value
            + 1e-8
        )
    )

    return heatmap.numpy()


def prepare_roi_for_classifier(
    roi_pil,
):
    resized = (
        roi_pil
        .convert("RGB")
        .resize(
            (
                CLS_IMAGE_SIZE[1],
                CLS_IMAGE_SIZE[0],
            ),
            Image.Resampling.BILINEAR,
        )
    )

    array = (
        np.asarray(resized)
        .astype("float32")
    )

    return (
        resized,
        array[None, ...],
    )


def heatmap_overlay(
    image_pil,
    heatmap,
    alpha=0.40,
):
    image = image_pil.convert(
        "RGB"
    )

    heatmap_resized = (
        Image.fromarray(
            np.uint8(
                np.clip(
                    heatmap,
                    0,
                    1,
                )
                * 255
            )
        )
        .resize(
            image.size,
            Image.Resampling.BILINEAR,
        )
    )

    heat = (
        np.asarray(
            heatmap_resized
        )
        .astype(
            np.float32
        )
        / 255.0
    )

    base = (
        np.asarray(image)
        .astype(
            np.float32
        )
        / 255.0
    )

    cmap = plt.get_cmap(
        "jet"
    )

    coloured = cmap(
        heat
    )[..., :3]

    overlay = np.clip(
        (
            (1 - alpha)
            * base
        )
        + (
            alpha
            * coloured
        ),
        0,
        1,
    )

    return Image.fromarray(
        np.uint8(
            overlay
            * 255
        )
    )


print(
    "Grad-CAM helper ready."
)

## 33. Grad-CAM examples

In [ ]:
gradcam_examples = []

for outcome in [
    "TP",
    "TN",
    "FP",
    "FN",
]:
    subset = (
        test_predictions_df[
            test_predictions_df[
                "outcome"
            ] == outcome
        ]
    )

    if len(subset):
        gradcam_examples.append(
            subset.iloc[0]
        )

for row in gradcam_examples:
    roi_pil = load_rgb(
        row["roi_path"]
    )

    (
        roi_resized,
        roi_batch,
    ) = prepare_roi_for_classifier(
        roi_pil
    )

    heatmap = make_gradcam_heatmap(
        final_classifier,
        tf.convert_to_tensor(
            roi_batch
        ),
    )

    overlay = heatmap_overlay(
        roi_resized,
        heatmap,
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(12, 4),
    )

    axes[0].imshow(
        load_rgb(
            row["image_path"]
        )
    )
    axes[0].set_title(
        f"Original — "
        f"{row['outcome']}"
    )
    axes[0].axis(
        "off"
    )

    axes[1].imshow(
        roi_resized
    )
    axes[1].set_title(
        "Lesion ROI"
    )
    axes[1].axis(
        "off"
    )

    axes[2].imshow(
        overlay
    )
    axes[2].set_title(
        f"Grad-CAM | "
        f"P(malignant)="
        f"{row['malignancy_probability']:.3f}"
    )
    axes[2].axis(
        "off"
    )

    plt.tight_layout()
    plt.show()

# PART F — END-TO-END INFERENCE AND GRADIO APP

The app accepts one dermoscopic image and returns:

1. original image,
2. segmentation overlay,
3. lesion ROI,
4. Grad-CAM overlay,
5. model probability of malignancy and thresholded output.

**Research/educational prototype only.**

## 34. End-to-end inference function

In [ ]:
def segmentation_overlay(
    image_pil,
    binary_mask,
    alpha=0.30,
):
    image = image_pil.convert(
        "RGB"
    )

    image_array = (
        np.asarray(image)
        .astype(
            np.float32
        )
    )

    mask_pil = (
        Image.fromarray(
            np.uint8(
                binary_mask
                * 255
            )
        )
        .resize(
            image.size,
            Image.Resampling.NEAREST,
        )
    )

    mask = (
        np.asarray(mask_pil)
        .astype(
            np.float32
        )
        / 255.0
    )

    overlay = (
        image_array.copy()
    )

    overlay[..., 0] = np.clip(
        (
            overlay[..., 0]
            * (1 - alpha)
        )
        + (
            255
            * mask
            * alpha
        ),
        0,
        255,
    )

    return Image.fromarray(
        np.uint8(
            overlay
        )
    )


def predict_melanoma(image):
    if image is None:
        raise ValueError(
            "Please upload a "
            "dermoscopic image."
        )

    if not isinstance(
        image,
        Image.Image,
    ):
        image = Image.fromarray(
            np.asarray(image)
        )

    image = image.convert(
        "RGB"
    )

    _, binary_mask = (
        predict_mask_for_pil(
            selected_segmenter,
            image,
        )
    )

    (
        roi,
        bbox,
        fallback,
    ) = extract_roi_from_binary_mask(
        image,
        binary_mask,
        padding_fraction=0.10,
    )

    (
        roi_resized,
        roi_batch,
    ) = prepare_roi_for_classifier(
        roi
    )

    probability = float(
        final_classifier
        .predict(
            roi_batch,
            verbose=0,
        )[0, 0]
    )

    prediction = (
        "Malignant / melanoma flagged"
        if probability >= SELECTED_CLS_THRESHOLD
        else "Benign / non-malignant predicted"
    )

    heatmap = make_gradcam_heatmap(
        final_classifier,
        tf.convert_to_tensor(
            roi_batch
        ),
    )

    gradcam_image = heatmap_overlay(
        roi_resized,
        heatmap,
    )

    mask_overlay = (
        segmentation_overlay(
            image,
            binary_mask,
        )
    )

    summary = (
        f"Model probability of malignancy: "
        f"{probability:.1%}\n"
        f"Thresholded model output: "
        f"{prediction}\n"
        f"Classification threshold: "
        f"{SELECTED_CLS_THRESHOLD:.3f}\n"
        f"ROI fallback used: "
        f"{fallback}\n\n"
        "Research/educational prototype only. "
        "This output is not a medical diagnosis "
        "and must not replace assessment by a "
        "qualified healthcare professional."
    )

    return {
        "original": image,
        "predicted_mask": Image.fromarray(np.uint8(binary_mask * 255)),
        "segmentation_overlay": (
            mask_overlay
        ),
        "roi": roi_resized,
        "gradcam": gradcam_image,
        "probability": probability,
        "prediction": prediction,
        "summary": summary,
        "bbox": bbox,
        "roi_fallback": fallback,
    }


print(
    "End-to-end prediction "
    "function ready."
)

## 35. Launch Gradio app

In [ ]:
import gradio as gr


def gradio_predict(image):
    result = predict_melanoma(
        image
    )

    return (
        result["original"],
        result[
            "segmentation_overlay"
        ],
        result["roi"],
        result["gradcam"],
        result["summary"],
    )


with gr.Blocks(
    title=(
        "Melanoma AI "
        "Research Prototype"
    )
) as demo:
    gr.Markdown(
        """
        # AI-Assisted Melanoma Analysis — Research Prototype

        Upload a **dermoscopic image**.

        The system displays:
        - lesion segmentation,
        - lesion ROI,
        - EfficientNetB0 malignancy probability,
        - Grad-CAM visualisation.

        **This is an educational/research prototype and is not a medical diagnosis.**
        """
    )

    input_image = gr.Image(
        type="pil",
        label=(
            "Upload dermoscopic image"
        ),
    )

    run_button = gr.Button(
        "Analyse image"
    )

    with gr.Row():
        original_output = gr.Image(
            label="Original"
        )
        segmentation_output = gr.Image(
            label="Segmentation overlay"
        )

    with gr.Row():
        roi_output = gr.Image(
            label="Lesion ROI"
        )
        gradcam_output = gr.Image(
            label="Grad-CAM"
        )

    summary_output = gr.Textbox(
        label="Model output",
        lines=8,
    )

    run_button.click(
        fn=gradio_predict,
        inputs=input_image,
        outputs=[
            original_output,
            segmentation_output,
            roi_output,
            gradcam_output,
            summary_output,
        ],
    )

demo.launch(
    share=True
)

# Skin-Tone Diversity and Fairness Limitation

ISIC 2016 Task 3B does not provide reliable skin-tone labels for a scientifically valid subgroup analysis. The notebook therefore does not invent Fitzpatrick labels, use brightness as a proxy, infer ethnicity from background colour, or claim that artificially darkening images solves demographic bias.

Future mitigation requires genuinely diverse dermoscopic data, expert subgroup annotation, subgroup validation, external validation, and reweighting only when valid group labels exist.

In [ ]:
def evaluate_fairness_by_group(predictions_dataframe, group_column):
    required = {"label", "predicted_label", "malignancy_probability", group_column}
    missing = required - set(predictions_dataframe.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    rows = []
    for group_value, group_df in predictions_dataframe.groupby(group_column):
        y_true_g = group_df["label"].astype(int).values
        y_pred_g = group_df["predicted_label"].astype(int).values
        probs_g = group_df["malignancy_probability"].astype(float).values

        cm_g = confusion_matrix(y_true_g, y_pred_g, labels=[0,1])
        tn_g, fp_g, fn_g, tp_g = cm_g.ravel()

        rows.append({
            "group": group_value,
            "n": len(group_df),
            "sensitivity": tp_g/(tp_g+fn_g) if (tp_g+fn_g) else np.nan,
            "specificity": tn_g/(tn_g+fp_g) if (tn_g+fp_g) else np.nan,
            "accuracy": accuracy_score(y_true_g, y_pred_g),
            "f1": f1_score(y_true_g, y_pred_g, zero_division=0),
            "roc_auc": roc_auc_score(y_true_g, probs_g)
            if len(np.unique(y_true_g)) == 2 else np.nan,
        })

    return pd.DataFrame(rows)

print("Fairness helper defined but not run: valid skin-tone labels are unavailable.")

# Other Challenges and Limitations

- Small sample size in `tutor_200` mode.
- Benign/malignant class imbalance.
- Limited skin-tone representation and no reliable subgroup labels.
- Dermoscopic-image domain differs from ordinary smartphone photography.
- Hair, bubbles, ruler markings, reflections and low contrast can affect segmentation.
- Segmentation errors can propagate downstream into the ROI and classifier.
- False positives and false negatives remain possible; melanoma false negatives are particularly important.
- Grad-CAM highlights influential regions but does not prove clinical causality.
- The binary dataset does not represent all dermatological diseases.
- Clinical deployment would require extensive external validation and regulatory review.

# 37. References

1. Gutman, D., Codella, N. C. F., Celebi, E., Helba, B., Marchetti, M., Mishra, N., & Halpern, A. (2016). *Skin Lesion Analysis toward Melanoma Detection: A Challenge at the International Symposium on Biomedical Imaging (ISBI) 2016, hosted by the International Skin Imaging Collaboration (ISIC).* arXiv:1605.01397.

2. Ronneberger, O., Fischer, P., & Brox, T. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation.* MICCAI.

3. Oktay, O., et al. (2018). *Attention U-Net: Learning Where to Look for the Pancreas.* arXiv:1804.03999.

4. Tan, M., & Le, Q. V. (2019). *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks.* ICML.

5. Selvaraju, R. R., et al. (2017). *Grad-CAM: Visual Explanations from Deep Networks via Gradient-Based Localization.* ICCV.

6. International Skin Imaging Collaboration (ISIC), **2016 Task 3B: Segmented Lesion Classification**.

7. Zhang, Z., Liu, Q., & Wang, Y. (2018). *Road Extraction by Deep Residual U-Net.* IEEE Geoscience and Remote Sensing Letters, 15(5), 749–753.

8. Daneshjou, R., et al. (2022). *Disparities in dermatology AI performance on a diverse, curated clinical image set.* Science Advances, 8(32).

## Save requested final aliases

In [ ]:
selected_segmenter.save(MODELS_DIR / "selected_segmenter.keras")
final_classifier.save(MODELS_DIR / "selected_classifier.keras")

if FINE_MODEL_PATH.exists():
    shutil.copy2(FINE_MODEL_PATH, MODELS_DIR / "efficientnet_b0_best.keras")
elif HEAD_MODEL_PATH.exists():
    shutil.copy2(HEAD_MODEL_PATH, MODELS_DIR / "efficientnet_b0_best.keras")

print("Saved requested model aliases and result tables to:", OUTPUT_DIR)

# Final workflow summary

```text
ISIC 2016 Task 3B
image + expert mask + benign/malignant label
                    │
                    ▼
             preprocessing
                    │
                    ▼
 Standard U-Net vs Attention U-Net vs ResUNet
                    │
      validation Dice + IoU comparison
                    │
                    ▼
          select best segmenter
                    │
                    ▼
            predicted lesion mask
                    │
                    ▼
                lesion ROI
                    │
                    ▼
        EfficientNetB0 classifier
            optional MobileNetV2
                    │
     validation-only model/threshold choice
                    │
                    ▼
              FINAL TEST ONLY
                    │
    TP / TN / FP / FN
    sensitivity / specificity
    accuracy / F1 / ROC-AUC / PR-AUC
                    │
                    ▼
                 Grad-CAM
                    │
                    ▼
                Gradio app
```

Daisy's U-Net++ metrics can be entered into the generated team comparison table when available.